In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# テンソルネットワーク誤り軽減（TEM）：AlgorithmiqによるQiskit Function
> **Note:** Qiskit Functionsは、IBM Quantum&reg; Premium Plan、Flex Plan、およびOn-Prem（IBM Quantum Platform API経由）Planユーザーのみが利用できる実験的な機能です。プレビューリリース状態であり、変更される可能性があります。


<details>
<summary><b>パッケージバージョン</b></summary>

このページのコードは以下の要件を使用して開発されました。
これらのバージョンまたはそれ以降の使用を推奨します。

```
qiskit[all]~=2.3.0
qiskit-ibm-catalog~=0.11.0
```
</details>
## 概要
Algorithmiqのテンソルネットワーク誤り軽減（TEM）手法は、古典的な後処理段階
でのみノイズ軽減を実行するために設計されたハイブリッド量子古典アルゴリズム
です。TEMを使用すると、量子ハードウェアで不可避的に発生するノイズ誘発エラーを
軽減しながら、オブザーバブルの期待値をより高い精度とコスト効率で計算できるため、
量子研究者および産業界の実務者にとって非常に魅力的な選択肢となっています。

この手法は、量子プロセッサの状態に影響を与えるグローバルノイズチャネルの逆を
表すテンソルネットワークを構築し、そのマップをノイジーな状態から取得した
情報完全な測定結果に適用して、オブザーバブルの不偏推定量を得るというものです。

利点として、TEMは情報完全な測定を活用して膨大なセットの軽減済みオブザーバブル
期待値へのアクセスを提供し、Filippov et al.（2023）、
[arXiv:2307.11740](https://arxiv.org/abs/2307.11740)、およびFilippov
et al.（2024）、[arXiv:2403.13542](https://arxiv.org/abs/2403.13542)で
説明されているように、量子ハードウェアでの最適なサンプリングオーバーヘッドを
実現します。測定オーバーヘッドとは、効率的な誤り軽減を実行するために必要な
追加測定の数を指し、量子計算の実現可能性における重要な要素です。したがって、
TEMは量子カオス、多体物理、ハバードダイナミクス、小分子化学シミュレーション
などの分野のアプリケーションにおいて、複雑なシナリオでの量子優位性を実現する
可能性を持っています。

TEMの主な特徴と利点は以下のようにまとめられます：

1.  **最適な測定オーバーヘッド**：TEMは
[理論的限界](https://journals.aps.org/prl/abstract/10.1103/PhysRevLett.131.210601)
に対して最適であり、どの手法もこれより小さい測定オーバーヘッドを達成できない
ことを意味します。言い換えれば、TEMは誤り軽減を実行するために必要な追加測定の
数が最小です。これは結果として、TEMが最小の量子ランタイムを使用することを
意味します。
2.  **コスト効率**：TEMは後処理段階でノイズ軽減を完全に処理するため、量子
コンピュータに追加の回路を投入する必要がなく、計算コストを削減するだけでなく、
量子デバイスの不完全性による追加エラーの導入リスクも低減します。
3.  **複数オブザーバブルの推定**：情報完全な測定のおかげで、TEMは量子
コンピュータからの同じ測定データで複数のオブザーバブルを効率的に推定します。
4.  **測定誤り軽減**：TEM Qiskit Functionには、短いキャリブレーション実行後に
読み取りエラーを大幅に削減できる
[独自の測定誤り軽減手法](https://journals.aps.org/prresearch/abstract/10.1103/PhysRevResearch.5.033154)
も含まれています。
5.  **精度**：TEMはデジタル量子シミュレーションの精度と信頼性を大幅に向上させ、
量子アルゴリズムをより正確で信頼性の高いものにします。
## 説明
TEM関数を使用すると、最小のサンプリングオーバーヘッドで量子回路上の複数の
オブザーバブルの誤り軽減済み期待値を取得できます。回路は情報完全正値演算子値
測度（IC-POVM）で測定され、収集された測定結果は古典コンピュータで処理されます。
この測定はテンソルネットワーク手法を実行し、ノイズ逆変換マップを構築するために
使用されます。関数はテンソルネットワークを使用してノイジーな層を表現し、
ノイジーな回路全体を完全に逆変換するマップを適用します。

![TEMの模式図](../docs/images/guides/algorithmiq-tem/tem_scheme.svg "ノイジーな量子プロセッサの測定結果の後処理によるオブザーバブルOの誤り軽減済み推定。UとNはそれぞれ理想的な量子演算と関連するノイズマップを示し、一般に非局所的（灰色のボックスに拡張）になりえます。DはIC測定のエフェクトに双対な演算子のテンソルです。ノイズ軽減モジュールMは、中央から外側に向かって効率的に縮約されるテンソルネットワークです。縮約の最初の反復は点線の紫の線、2番目は破線、3番目は実線で表されています。")

回路が関数に送信されると、トランスパイルおよび最適化が行われ、2量子ビット
ゲート（量子デバイスでよりノイジーなゲート）を含む層の数が最小化されます。
層に影響を与えるノイズは、E. van den Berg, Z. Minev, A. Kandala, K. Temme,
Nat. Phys.（2023）、
[arXiv:2201.09866](https://arxiv.org/abs/2201.09866)で説明されているスパース
Pauli-Lindbladノイズモデルを使用して
[Qiskit Runtime](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/noise-learner-noise-learner)
を通じて学習されます。

このノイズモデルは、量子ビットのクロストークなどの微妙な特徴を捉えることが
できる、デバイス上のノイズの正確な記述です。ただし、デバイス上のノイズは
変動やドリフトする可能性があり、推定が行われる時点で学習されたノイズが正確で
ない場合があります。これにより不正確な結果が生じる可能性があります。
## はじめに
[IBM Quantum Platform APIキー](http://quantum.cloud.ibm.com/)を使用して認証し、以下のようにTEM関数を選択します。（このスニペットでは、既に[アカウントをローカル環境に保存](/guides/functions#install-qiskit-functions-catalog-client)していることを前提としています。）

In [1]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

tem_function_name = "algorithmiq/tem"

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# Load your function
tem = catalog.load(tem_function_name)

## 例
以下のスニペットは、TEMを使用して単純な量子回路に対するオブザーバブルの期待値を計算する例を示しています。

In [2]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp

# Create a quantum circuit
qc = QuantumCircuit(3)
qc.u(0.4, 0.9, -0.3, 0)
qc.u(-0.4, 0.2, 1.3, 1)
qc.u(-1.2, -1.2, 0.3, 2)
for _ in range(2):
    qc.barrier()
    qc.cx(0, 1)
    qc.cx(2, 1)
    qc.barrier()
    qc.u(0.4, 0.9, -0.3, 0)
    qc.u(-0.4, 0.2, 1.3, 1)
    qc.u(-1.2, -1.2, 0.3, 2)

# Define the observables
observable = SparsePauliOp("IYX", 1.0)

# Define the execution options
pub = (qc, [observable])
options = {"default_precision": 0.02}

# Define backend to use. TEM will choose the least-busy device reported by IBM if not specified
backend_name = "ibm_torino"

job = tem.run(pubs=[pub], backend_name=backend_name, options=options)

Qiskit Serverless APIを使用してQiskit Functionワークロードのステータスを確認します：

In [3]:
print(job.status())

QUEUED


You can return the results as:

In [4]:
result = job.result()
evs = result[0].data.evs

結果は以下のように取得できます：

In [5]:
print(job.result())

PrimitiveResult([PubResult(data=DataBin(evs=np.ndarray(<shape=(1,), dtype=float64>), stds=np.ndarray(<shape=(1,), dtype=float64>)), metadata={'evs_non_mitigated': array([-0.06314623]), 'stds_non_mitigated': array([0.05917147]), 'evs_mitigated_no_readout_mitigation': array([-0.06411205]), 'stds_mitigated_no_readout_mitigation': array([0.05992467]), 'evs_non_mitigated_with_readout_mitigation': array([-0.07028881]), 'stds_non_mitigated_with_readout_mitigation': array([0.06353934])})], metadata={'resource_usage': {'RUNNING: OPTIMIZING_FOR_HARDWARE': {'CPU_TIME': 0.915754}, 'RUNNING: WAITING_FOR_QPU': {'CPU_TIME': 18.804865}, 'RUNNING: POST_PROCESSING': {'CPU_TIME': 10.433445}, 'RUNNING: EXECUTING_QPU': {'QPU_TIME': 159.0}}})


> **Info:** ノイズなし回路での指定された演算子に対する期待値は、おおよそ`0.18409094298943401`になるはずです。
## 入力
**パラメータ**

Name | Type | Description | Required | Default | Example
-- | -- | -- | -- | -- | --
`pubs` | Iterable[EstimatorPubLike] | PUBライク（primitive unified bloc）オブジェクトの反復可能オブジェクト。タプル`(circuit, observables)`または`(circuit, observables, parameters, precision)`など。詳細は[PUBの概要](/guides/primitive-input-output#overview-of-pubs)を参照してください。非ISA回路が渡された場合、最適な設定でトランスパイルされます。ISA回路が渡された場合、トランスパイルは行われません。この場合、オブザーバブルはQPU全体で定義される必要があります。 | Yes | N/A | (circuit, observables)
`backend_name` | str | クエリを実行するバックエンドの名前。| No | 提供されない場合、最も空いているバックエンドが使用されます。 | "ibm_fez"
`options` | dict | 入力オプション。詳細は`Options`セクションを参照してください。 | No | 詳細は`Options`セクションを参照してください。| {"max_bond_dimension": 100\}

> **Caution:** TEMには現在以下の制限があります：
> 
>   - パラメータ化された回路はサポートされていません。precisionが指定されている場合、parametersの引数は`None`に設定する必要があります。この制限は将来のバージョンで解除されます。
>   - ループのない回路のみがサポートされています。この制限は将来のバージョンで解除されます。
>   - reset、measure、およびすべての形式の制御フローなどの非ユニタリゲートはサポートされていません。resetのサポートは今後のリリースで追加されます。
### オプション
TEMの詳細オプションを含む辞書です。辞書には以下の表のキーを含めることができます。オプションが提供されない場合、表に記載されているデフォルト値が使用されます。デフォルト値はTEMの一般的な使用に適しています。

Name | Choices | Description  | Default
-- | -- | -- | --
`tem_max_bond_dimension` | int | テンソルネットワークに使用される最大ボンド次元。 | 500 |
`tem_compression_cutoff` | float | テンソルネットワークに使用されるカットオフ値。 | 1e-16
`compute_shadows_bias_from_observable` | bool | 古典シャドウ測定プロトコルのバイアスをPUBオブザーバブルに合わせて調整するかどうかを示すブールフラグ。Falseの場合、古典シャドウプロトコル（Z、X、Yを等確率で測定）が使用されます。| False
`shadows_bias` | np.ndarray | ランダム化古典シャドウ測定プロトコルに使用されるバイアス。サイズ3の1次元配列または形状(num_qubits, 3)の2次元配列。順序はZXY。 | np.array([1 / 3, 1 / 3, 1 / 3])
`max_execution_time` | int or `None` | QPU上の最大実行時間（秒）。ランタイムがこの値を超えると、ジョブはキャンセルされます。`None`の場合、Qiskit Runtimeによるデフォルトの制限が適用されます。 | `None`
`num_randomizations` | int | ノイズ学習とゲートトワーリングに使用されるランダム化の数。 | 32
`max_layers_to_learn` | int | 学習する一意の層の最大数。 | 4
`mitigate_readout_error` | bool | 読み取り誤り軽減を実行するかどうかを示すブールフラグ。 | True
`num_readout_calibration_shots` | int | 読み取り誤り軽減に使用されるショット数。 | 10000
`default_precision` | float | precisionが指定されていないPUBに使用されるデフォルトの精度。 |0.02
`seed` | int or `None` | 再現性のために乱数生成器のシードを設定します。`None`の場合、シードは設定されません。 | None
## 出力
TEM軽減済みの結果を含むQiskit [PrimitiveResults](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PrimitiveResult)。各PUBの結果は以下のフィールドを含む[PubResult](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PubResult)として返されます：

Name |Type | Description
-- | -- | --
data | DataBin | TEM軽減済みのオブザーバブルとその標準誤差を含むQiskit [DataBin](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.DataBin)。DataBinには以下のフィールドがあります：<ul><li>`evs`：TEM軽減済みのオブザーバブル値。</li><li>`stds`：TEM軽減済みオブザーバブルの標準誤差。</li></ul>
metadata | dict | 追加の結果を含む辞書。辞書には以下のキーが含まれます：<ul><li>`"evs_non_mitigated"`：誤り軽減なしのオブザーバブル値。</li><li>`"stds_non_mitigated"`：誤り軽減なしの結果の標準誤差。</li><li>`"evs_mitigated_no_readout_mitigation"`：誤り軽減ありだが読み取り誤り軽減なしのオブザーバブル値。</li><li>`"stds_mitigated_no_readout_mitigation"`：誤り軽減ありだが読み取り誤り軽減なしの結果の標準誤差。</li><li>`"evs_non_mitigated_with_readout_mitigation"`：誤り軽減なしだが読み取り誤り軽減ありのオブザーバブル値。</li><li>`"stds_non_mitigated_with_readout_mitigation"`：誤り軽減なしだが読み取り誤り軽減ありの結果の標準誤差。</li></ul>
## エラーメッセージの取得
ワークロードのステータスがERRORの場合、以下のようにjob.result()を使用してエラーメッセージを取得します：